# 08 · 官方算例复现

一键复现 DESY 官方 9 个算例 (Manual/Aperture/Wake/Cavity/Curved_Cathode/90deg_bend/Plasma_1/Plasma_2) 并与黄金样本比对。
注: Plasma_Example_2 需先下载 plasma_user_laser.zip 并解压到 examples/Plasma_Example_2/。

In [ ]:
%run _bootstrap.py

In [ ]:
# 官方 9 算例: 运行流程 + 黄金样本比对
import shutil, json
from pathlib import Path
from astra_tools.run import run_program
from astra_tools.io.astra_emit import parse_output_file
from astra_tools.io.field_map import fix_laser_map_header

EXAMPLES_DIR = PROJECT_ROOT / "examples"
GOLDEN_EXPECTED = json.loads((EXAMPLES_DIR / "golden_expected.json").read_text())

# 每个算例: 输入文件清单 / 运行步骤 / 黄金比对目标
EXAMPLES = {
    "Manual_Example": dict(
        copy=["generator.in", "Example.in", "3_cell_L-Band.dat", "Solenoid.dat"],
        rename={"Example.in": "astra.in"},
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Manual_Example/Example.Xemit.001"),
    "Aperture": dict(
        copy=["astra.in", "aperture.in", "Geometry.dat", "test.ini"],
        steps=[("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Aperture/golden/astra.Xemit.001"),
    "Wake": dict(
        copy=["Wake.in", "test.ini", "TESLA_MODULE_WAKE_TAYLOR.dat", "test.dat"],
        src_dir="Wake/Wake_Files",
        steps=[("astra", "Wake.in")],
        golden_xemit=EXAMPLES_DIR / "Wake/golden/Wake.Xemit.001"),
    "Cavity_Example": dict(
        copy=["generator.in", "astra.in", "TWS_Sband.dat", "3_cell_L-Band.dat",
              "dcfield.dat", "3D_test.bx", "3D_test.by", "3D_test.bz"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Cavity_Example/golden/astra.Xemit.001"),
    "Curved_Cathode_Example": dict(
        copy=["generator.in", "astra.in", "Contour.dat", "efld.dat"],
        steps=[("generator", "generator.in"), ("astra", "astra.in")],
        golden_xemit=EXAMPLES_DIR / "Curved_Cathode_Example/golden/astra.Xemit.001"),
    "90deg_bend_Example": dict(
        copy=["Section1.in", "Section2.in", "test.ini",
              "3D_Dipole.bx", "3D_Dipole.by", "3D_Dipole.bz"],
        patch={"Section2.in": ("Section1_n.0100.001", "Section1.0100.001")},
        steps=[("astra", "Section1.in"), ("astra", "Section2.in")],
        golden_xemit=EXAMPLES_DIR / "90deg_bend_Example/golden/Section2.Log.001",
        compare_mode="log"),
    "Plasma_Example_1": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt"],
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_1/golden/plasma.Xemit.001"),
    "Plasma_Example_2": dict(
        copy=["phsp.in", "plasma.in", "PLASMA_flattop.txt", "laser.dat"],
        laser_fix=True,
        steps=[("generator", "phsp.in"), ("astra", "plasma.in")],
        golden_xemit=EXAMPLES_DIR / "Plasma_Example_2/golden/plasma.Xemit.001"),
}


def run_example(name):
    spec = EXAMPLES[name]
    src = EXAMPLES_DIR / spec.get("src_dir", name)
    work = SIM_DIR / name
    work.mkdir(parents=True, exist_ok=True)
    for f in spec["copy"]:
        s = src / f
        if not s.exists():
            raise FileNotFoundError(
                "%s 缺失: %s\n   Plasma_Example_2 请从 DESY 下载 plasma_user_laser.zip 并解压"
                % (f, s))
        shutil.copy2(s, work / f)
    for deck, (old, new) in spec.get("patch", {}).items():
        p = work / deck
        p.write_text(p.read_text().replace(old, new))
    if spec.get("laser_fix"):
        fix_laser_map_header(work / "laser.dat")
        print("  laser.dat 3D 图头已转换为 ASTRA 网格格式")
    for kind, deck in spec["steps"]:
        exe = GENERATOR_EXE if kind == "generator" else ASTRA_EXE
        run_program(exe, work, input_file=deck)
    return work


def compare_xemit(name, work):
    spec = EXAMPLES[name]
    golden = spec["golden_xemit"]
    new_file = work / golden.name
    if not new_file.exists():
        print("  (无 %s 输出, 跳过比对)" % golden.name)
        return
    new = parse_output_file(new_file)
    ref = parse_output_file(golden)
    print("  末行比对 (new vs golden):")
    for key in ("norm_emit_x", "sigma_x", "mean_z"):
        a = float(__import__("numpy").asarray(new[key])[-1])
        b = float(__import__("numpy").asarray(ref[key])[-1])
        rel = abs(a - b) / abs(b) * 100
        print("    %-14s %-10.6g %-10.6g rel=%.4f%% %s"
              % (key, a, b, rel, "OK" if rel < 0.5 else "MISMATCH"))


In [ ]:
# 复现一个算例 (修改 EXAMPLE 名即可)
EXAMPLE = "Manual_Example"
work = run_example(EXAMPLE)
compare_xemit(EXAMPLE, work)

或一键运行全部 9 个算例 (约一分钟):

In [ ]:
# 一键复现全部 9 个官方算例
for name in EXAMPLES:
    print("=" * 60)
    print(name)
    try:
        work = run_example(name)
        compare_xemit(name, work)
    except Exception as e:
        print("  FAILED:", e)
    print()